# SHD reservoir-SNN walkthrough — from raw events to a two-way ridge test

A **self-contained, minimal** notebook (pure Python + PyTorch, *no snnTorch*) that goes
end-to-end on the **Spiking Heidelberg Digits (SHD)** dataset:

1. **Load** the raw event-based `.h5` files and **visualise** one utterance as events.
2. **Temporally bin** the events into dense spike tensors and visualise the result.
3. **Split** the official train set into train/val (test stays the **official** test set) and plot class histograms.
4. Define one **second-order recurrent LIF reservoir** with three interchangeable readouts.
5. **Train** it four ways: BPTT on (a) mean output-membrane, (b) max output-membrane, (c) firing-rate; and (d) **closed-form ridge** on the time-averaged hidden spikes.
6. **Test the ridge `W_out` two ways** on the official test set and explain *why the two numbers differ*.

Everything is driven by the single config cell below — change a constant, re-run.


## 1. Configuration

Every knob lives here so the rest of the notebook is just logic.

Two choices deserve a word:

* **Binning vs. simulation time-step are decoupled.** We keep `NB_STEPS=100` bins over a
  `MAX_TIME=1.4 s` window, so each bin aggregates ~14 ms of events. But the neuron decay
  is computed with `SIM_DT_MS = 1 ms`, following the Zenke SHD-tutorial convention. This is
  deliberate: the membrane decay per step is `BETA = exp(-1/10) ≈ 0.90`, giving the neuron a
  **long memory** that integrates evidence across the whole utterance. If instead you set
  `SIM_DT_MS = 14`, you get `BETA = exp(-14/10) ≈ 0.25` — the membrane forgets almost
  everything every step and accuracy collapses. The long effective time constant is what
  makes a temporal task like SHD learnable.
* **Spectral-radius rescaling** pushes the recurrent matrix to the "edge of chaos"
  (`SPECTRAL_RADIUS=0.8`), a standard reservoir trick that keeps the dynamics rich but stable.

`NB_HIDDEN` and `NB_EPOCHS` are set for a reasonable interactive run; the comments show the
values that approach paper-level accuracy.


In [ ]:
import math, time
import numpy as np
import torch, torch.nn as nn
import h5py
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

DEVICE = torch.device("mps"  if torch.backends.mps.is_available()
                 else "cuda" if torch.cuda.is_available()
                 else "cpu")
print("device:", DEVICE)

# --- data / temporal binning ---
DATA_DIR   = "data"          # folder holding shd_train.h5 / shd_test.h5
NB_INPUTS  = 700             # SHD input channels
NB_OUTPUTS = 20              # spoken digits 0-9 in English + German
MAX_TIME   = 1.4             # seconds kept per utterance
NB_STEPS   = 100             # number of time bins  ->  bin width = 14 ms
DT_S       = MAX_TIME / NB_STEPS

# --- neuron dynamics (simulate at 1 ms even though bins are ~14 ms) ---
SIM_DT_MS  = 1.0
TAU_SYN_MS = 5.0
TAU_MEM_MS = 10.0
ALPHA = math.exp(-SIM_DT_MS / TAU_SYN_MS)   # synaptic-current decay per step
BETA  = math.exp(-SIM_DT_MS / TAU_MEM_MS)   # membrane decay per step
THRESHOLD  = 1.0
SURR_SLOPE = 25.0                           # fast-sigmoid surrogate steepness

# --- weights ---
NB_HIDDEN       = 256        # paper uses 1000; smaller => faster BPTT
WEIGHT_SCALE    = 0.2        # gaussian init std = WEIGHT_SCALE / sqrt(fan_in)
SPECTRAL_RADIUS = 0.8        # rescale recurrent matrix to this radius; <=0 disables

# --- training ---
BATCH_SIZE   = 256
LR           = 2e-4
NB_EPOCHS    = 20            # bump to ~200 to approach paper accuracy
RIDGE_LAMBDA = 1.0

print(f"ALPHA={ALPHA:.3f}  BETA={BETA:.3f}  bin={DT_S*1000:.1f} ms  sim_dt={SIM_DT_MS} ms")

## 2. Load the raw event data

SHD ships as HDF5. Each sample is a **ragged list of spike events**: `spikes/times[i]` holds
the firing times (seconds) and `spikes/units[i]` the corresponding input-channel index, plus
one integer `labels[i]`. There is no fixed-size array yet — that is what binning (next) creates.

We load `shd_train.h5` and `shd_test.h5` separately and **keep them separate**: we train on the
official train set (with a val slice carved out) and report the final number on the **official
test set**, the benchmark-correct protocol.


In [ ]:
def load_pool(path):
    with h5py.File(path, "r") as f:
        times  = [np.asarray(t, np.float64) for t in f["spikes"]["times"]]
        units  = [np.asarray(u, np.int64)  for u in f["spikes"]["units"]]
        labels = np.asarray(f["labels"], np.int64)
    return times, units, labels

tr_times, tr_units, tr_labels = load_pool(f"{DATA_DIR}/shd_train.h5")
te_times, te_units, te_labels = load_pool(f"{DATA_DIR}/shd_test.h5")
print("train samples:", len(tr_labels), " | test samples:", len(te_labels))

## 3. Visualise one utterance as raw events

Before any processing, let's *see* the data. A spike raster scatter-plots every event of one
sample: time on the x-axis, input channel (0–699) on the y-axis. SHD channels are tonotopically
ordered (low index ≈ low frequency), so a spoken digit shows up as diagonal/structured streaks.


In [ ]:
i = 0
plt.figure(figsize=(7, 3))
plt.scatter(tr_times[i], tr_units[i], s=1, c="k", marker=".")
plt.xlabel("time (s)"); plt.ylabel("input channel (0-699)")
plt.title(f"Raw events - sample {i}, label {tr_labels[i]}  ({len(tr_times[i])} spikes)")
plt.xlim(0, MAX_TIME); plt.ylim(0, NB_INPUTS); plt.tight_layout(); plt.show()

## 4. Temporal binning (events -> dense spike tensor)

A network needs a fixed-size tensor per sample, so we discretise time into `NB_STEPS` bins of
width `DT_S`. Each spike at time `t` is placed in bin `floor(t / DT_S)`; spikes at
`t >= MAX_TIME` are dropped and the index is clipped to the last bin to absorb float edge cases.
The result is a dense `uint8` tensor `[N, NB_STEPS, NB_INPUTS]` of 0/1 (we treat a bin as "fired"
if it received ≥1 event).

We keep the tensors as `uint8` on CPU (compact, ~0.5 GB for train) and cast each *batch* to float
on the device on demand — this avoids holding a multi-GB float tensor on the GPU.

> Why `floor(t/dt)` and not `linspace`+`digitize`? `digitize` is 1-indexed and can return an
> out-of-range bin for `t≈max_time`; plain floor binning is 0-indexed, uniform, and crash-free.


In [ ]:
def bin_pool(times, units, labels):
    N = len(labels)
    X = torch.zeros((N, NB_STEPS, NB_INPUTS), dtype=torch.uint8)
    for n in range(N):
        t, u = times[n], units[n]
        keep = t < MAX_TIME
        t, u = t[keep], u[keep]
        b = np.clip((t / DT_S).astype(np.int64), 0, NB_STEPS - 1)
        X[n, torch.from_numpy(b), torch.from_numpy(u)] = 1
    return X, torch.from_numpy(labels).long()

X_train_full, y_train_full = bin_pool(tr_times, tr_units, tr_labels)
X_test,       y_test       = bin_pool(te_times, te_units, te_labels)
print("binned train:", tuple(X_train_full.shape), " | test:", tuple(X_test.shape))

## 5. Visualise the same utterance after binning

Same sample as in step 3, now on the discrete bin grid. Compared to the raw raster it is coarser
in time (100 columns) and binary per (bin, channel) — this is exactly what the network sees.


In [1]:
i = 0
ts, cs = np.nonzero(X_train_full[i].numpy())   # (time_bins, channels) that fired
plt.figure(figsize=(7, 3))
plt.scatter(ts, cs, s=1, c="k", marker=".")
plt.xlabel(f"time bin (0-{NB_STEPS-1},  {DT_S*1000:.0f} ms each)"); plt.ylabel("input channel")
plt.title(f"Binned spikes - sample {i}, label {tr_labels[i]}")
plt.xlim(0, NB_STEPS); plt.ylim(0, NB_INPUTS); plt.tight_layout(); plt.show()

NameError: name 'np' is not defined

## 6. Train/val split + class histograms

The official test set is untouched. We carve a **15% validation** slice out of the official train
set, **stratified by class** (each class keeps its proportion) so val is never starved of a digit.
The histograms below confirm all 20 classes are present and roughly balanced in every split.


In [ ]:
def stratified_val_split(y, val_frac=0.15, seed=0):
    rng = np.random.default_rng(seed); y = y.numpy(); tr, va = [], []
    for c in range(NB_OUTPUTS):
        idx = np.where(y == c)[0]; rng.shuffle(idx)
        k = int(round(len(idx) * val_frac))
        va.extend(idx[:k]); tr.extend(idx[k:])
    return np.array(sorted(tr)), np.array(sorted(va))

tr_idx, va_idx = stratified_val_split(y_train_full, 0.15, SEED)
X_tr, y_tr = X_train_full[tr_idx], y_train_full[tr_idx]
X_va, y_va = X_train_full[va_idx], y_train_full[va_idx]
print(f"train {len(y_tr)} | val {len(y_va)} | test {len(y_test)}")

fig, ax = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for a, (name, yy) in zip(ax, [("train", y_tr), ("val", y_va), ("test", y_test)]):
    a.bar(np.arange(NB_OUTPUTS), np.bincount(yy.numpy(), minlength=NB_OUTPUTS))
    a.set_title(name); a.set_xlabel("class")
ax[0].set_ylabel("count"); plt.tight_layout(); plt.show()

## 7. The spike function and its surrogate gradient

A spike is a hard threshold: `s = 1 if (mem - threshold) > 0 else 0`. That step has zero gradient
almost everywhere, so BPTT can't learn through it. The standard fix is a **surrogate gradient**:
the forward pass uses the true Heaviside step, but the backward pass pretends the step was a smooth
*fast sigmoid*, whose derivative is `1 / (slope·|x| + 1)²`. Larger `SURR_SLOPE` ⇒ sharper, more
spike-like surrogate. We implement this as a custom `autograd.Function`.


In [ ]:
class SpikeFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).float()                       # true Heaviside (forward)
    @staticmethod
    def backward(ctx, grad_out):
        x, = ctx.saved_tensors
        return grad_out / (SURR_SLOPE * x.abs() + 1.0) ** 2   # fast-sigmoid surrogate

spike = SpikeFn.apply

## 8. The reservoir SNN

One recurrent hidden layer of **second-order LIF** neurons, plus a readout matrix `W_out`.

**Per time step** the hidden neuron integrates input current `cur` (feed-forward + recurrent):

```
syn = ALPHA * syn + cur          # synaptic current (1st order, decay ALPHA)
mem = BETA  * mem + syn          # membrane potential (2nd order, decay BETA)
spk = spike(mem - THRESHOLD)     # emit a spike where mem crosses threshold
mem = mem * (1 - spk)            # reset-to-zero after a spike
```

`hidden(x)` returns the full spike trace `[B, T, H]`.

For the membrane-based readouts we add a **non-spiking output neuron** (`output_lif`): it runs the
*same* `ALPHA/BETA` dynamics but **never spikes and never resets**, so it simply low-pass-filters
the projected hidden spikes `spk @ W_out` and returns its membrane trace `[B, T, 20]`. This is the
"output LIF head". The firing-rate readout skips it entirely.

`make_net(seed)` builds parameters on CPU under a fixed seed (so every method below starts from the
**identical** random reservoir) then moves to the device. Init std is `WEIGHT_SCALE/sqrt(fan_in)`,
and the recurrent matrix is rescaled to `SPECTRAL_RADIUS` (computed on CPU; `eigvals` is unsupported
on MPS).


In [ ]:
class ReservoirSNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_in  = nn.Parameter(torch.randn(NB_INPUTS,  NB_HIDDEN)  * (WEIGHT_SCALE / math.sqrt(NB_INPUTS)))
        self.W_rec = nn.Parameter(torch.randn(NB_HIDDEN,  NB_HIDDEN)  * (WEIGHT_SCALE / math.sqrt(NB_HIDDEN)))
        self.W_out = nn.Parameter(torch.randn(NB_HIDDEN,  NB_OUTPUTS) * (WEIGHT_SCALE / math.sqrt(NB_HIDDEN)))
        if SPECTRAL_RADIUS > 0:                                   # edge-of-chaos rescale (on CPU)
            with torch.no_grad():
                r = torch.linalg.eigvals(self.W_rec).abs().max()
                self.W_rec.mul_(SPECTRAL_RADIUS / r)

    def hidden(self, x):                                          # x:[B,T,C] -> spikes [B,T,H]
        B = x.shape[0]
        h_in = x @ self.W_in                                     # feed-forward drive, all steps
        syn = torch.zeros(B, NB_HIDDEN, device=x.device)
        mem = torch.zeros_like(syn); spk = torch.zeros_like(syn); rec = []
        for t in range(NB_STEPS):
            cur = h_in[:, t] + spk @ self.W_rec                  # + recurrent drive from last step
            syn = ALPHA * syn + cur
            mem = BETA  * mem + syn
            spk = spike(mem - THRESHOLD)
            mem = mem * (1.0 - spk.detach())                     # reset-to-zero
            rec.append(spk)
        return torch.stack(rec, dim=1)

    def output_lif(self, spk):                                   # spikes -> output Vmem [B,T,20]
        h = spk @ self.W_out                                     # per-step projection to classes
        syn = torch.zeros(h.shape[0], NB_OUTPUTS, device=h.device)
        mem = torch.zeros_like(syn); rec = []
        for t in range(NB_STEPS):
            syn = ALPHA * syn + h[:, t]
            mem = BETA  * mem + syn                              # non-spiking: never threshold/reset
            rec.append(mem)
        return torch.stack(rec, dim=1)

    def forward(self, x, need_output=True):                      # snnTorch-style trace dict
        x = x.float()
        hidden_spike_trace = self.hidden(x)
        traces = {"hidden_spike_trace": hidden_spike_trace}
        if need_output:
            traces["output_membrane_trace"] = self.output_lif(hidden_spike_trace)
        return traces

def make_net(seed=SEED):
    torch.manual_seed(seed)
    return ReservoirSNN().to(DEVICE)

## 9. The three readouts, batching, and evaluation

Following the snnTorch / `train_shd_*` convention, `net(x)` runs the reservoir once and returns a
**trace dict** (`hidden_spike_trace`, `output_membrane_trace`). `readout_logits(net, traces, mode)`
then collapses time into **20 logits** three ways:

| `mode`        | logits                                   | uses output LIF? |
|---------------|------------------------------------------|------------------|
| `firing_rate` | `mean_t(spikes) @ W_out`                  | no (pure linear) |
| `mean_vmem`   | `mean_t(output_LIF(spikes @ W_out))`     | yes              |
| `max_vmem`    | `max_t (output_LIF(spikes @ W_out))`     | yes              |

`accuracy` calls `readout_logits` and argmaxes — **one evaluation function reused everywhere**, so any
difference between runs is purely the readout/training.


In [ ]:
def reduce_output_membrane(output_mem_trace, reduce):
    if reduce == "mean":
        return output_mem_trace.mean(dim=1)
    if reduce == "max":
        return output_mem_trace.max(dim=1).values
    raise ValueError(f"unknown reduce: {reduce!r}")

def readout_logits(net, traces, mode):
    """Form [B, NB_OUTPUTS] logits from a forward trace dict."""
    if mode == "firing_rate":
        return traces["hidden_spike_trace"].mean(dim=1) @ net.W_out
    reduce = "mean" if mode == "mean_vmem" else "max"
    return reduce_output_membrane(traces["output_membrane_trace"], reduce)

def batches(X, y, bs, shuffle, device=DEVICE):
    idx = torch.randperm(len(y)) if shuffle else torch.arange(len(y))
    for s in range(0, len(y), bs):
        j = idx[s:s + bs]
        yield X[j].float().to(device), y[j].to(device)    # cast uint8->float on device per batch

@torch.no_grad()
def accuracy(net, X, y, mode):
    net.eval()
    correct = total = 0
    need_output = mode != "firing_rate"
    for xb, yb in batches(X, y, BATCH_SIZE, shuffle=False):
        logits = readout_logits(net, net(xb, need_output=need_output), mode)
        correct += int((logits.argmax(dim=1) == yb).sum().item())
        total += int(yb.numel())
    return correct / max(total, 1)

## 10. A single BPTT trainer

Backprop-through-time over the whole recurrent network (`W_in`, `W_rec`, `W_out` all trainable),
cross-entropy on the 20-class logits, Adamax optimiser — the same recipe for all three gradient
readouts, selected by `mode`. We start from a fresh identical reservoir (`make_net(seed)`) so the
three runs are directly comparable.

> The BPTT cells are the slow part (a recurrent loop of `NB_STEPS` per batch). For a quick pass keep
> `NB_HIDDEN`/`NB_EPOCHS` modest; for accuracy near the reference scripts raise them (1000 / ~200).


In [ ]:
def train_bptt(mode, epochs=NB_EPOCHS, seed=SEED, verbose_every=5):
    net = make_net(seed)
    opt = torch.optim.Adamax(net.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    for ep in range(epochs):
        net.train(); t0 = time.time()
        for xb, yb in batches(X_tr, y_tr, BATCH_SIZE, shuffle=True):
            logits = readout_logits(net, net(xb, need_output=(mode != "firing_rate")), mode)
            loss = loss_fn(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep + 1) % verbose_every == 0 or ep == 0:
            print(f"[{mode}] epoch {ep+1:3d}/{epochs}  val_acc={accuracy(net, X_va, y_va, mode):.4f}"
                  f"  ({time.time()-t0:.1f}s/epoch)")
    return net

## 11. BPTT — mean output-membrane readout

Train end-to-end with logits = **time-average of the output-LIF membrane**. This is a smooth,
fully-differentiable readout and the natural partner of the ridge solution later.


In [ ]:
net_mean = train_bptt("mean_vmem")
print("TEST  mean_vmem :", round(accuracy(net_mean, X_test, y_test, "mean_vmem"), 4))

## 12. BPTT — max output-membrane readout

Same network, but logits = **peak over time of the output-LIF membrane**. Intuitively this asks
"at the most confident instant, how high did each class get?" — often stronger than the mean on
SHD because the spoken word occupies only part of the window. Gradient descent handles the `max`
fine (it routes the gradient to the winning time step).


In [ ]:
net_max = train_bptt("max_vmem")
print("TEST  max_vmem  :", round(accuracy(net_max, X_test, y_test, "max_vmem"), 4))

## 13. BPTT — firing-rate readout

No output neuron at all: logits = **mean firing rate per hidden neuron** projected by `W_out`.
This is the simplest possible readout and is exactly the *same functional form* that ridge will
solve in closed form next — the only difference there will be **how `W_out` is obtained**
(gradient descent here vs. one linear solve).


In [ ]:
net_rate = train_bptt("firing_rate")
print("TEST  firing_rate:", round(accuracy(net_rate, X_test, y_test, "firing_rate"), 4))

## 14. Ridge regression — closed-form `W_out` on time-averaged hidden spikes

Now **freeze the reservoir** (random `W_in`, `W_rec`) and solve for `W_out` in one shot. The feature
for each sample is the time-averaged hidden spike vector `Φ = mean_t(spikes)` (shape `[N, H]`); the
target is the one-hot label over all 20 classes. Ridge minimises `‖ΦW − Y‖² + λ‖W‖²`, whose solution
is the **normal equations**

$$W = (Φ^\top Φ + λI)^{-1} Φ^\top Y.$$

This is valid precisely because the prediction `Φ @ W` is **linear in `W`** — a fixed feature matrix.
(A `max`-over-time readout would *not* be linear in `W`, so it has no closed form; that is why ridge
always pairs with the mean/rate readout, never `max`.)

We run the linear algebra on **CPU in float64** for numerical stability (and because MPS lacks
float64), then copy the result back into `net_ridge.W_out`.


In [ ]:
net_ridge = make_net(SEED)                         # SAME random reservoir as the BPTT runs
for p in net_ridge.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def collect_mean_spikes(net, X, y):
    net.eval(); feats = []
    for xb, _ in batches(X, y, BATCH_SIZE, shuffle=False):
        feats.append(net(xb, need_output=False)["hidden_spike_trace"].mean(dim=1).cpu())  # [b,H]
    return torch.cat(feats)                              # [N,H]

Phi = collect_mean_spikes(net_ridge, X_tr, y_tr).double()              # [N,H]  (CPU, float64)
Y   = torch.zeros(len(y_tr), NB_OUTPUTS, dtype=torch.float64)
Y[torch.arange(len(y_tr)), y_tr] = 1.0                                 # one-hot, 20 classes
A = Phi.T @ Phi + RIDGE_LAMBDA * torch.eye(NB_HIDDEN, dtype=torch.float64)
W = torch.linalg.solve(A, Phi.T @ Y)                                   # [H,20]
with torch.no_grad():
    net_ridge.W_out.copy_(W.float().to(DEVICE))
print("ridge train acc (firing_rate):", round(accuracy(net_ridge, X_tr, y_tr, "firing_rate"), 4))

## 15. Test the ridge `W_out` — WAY 1: exactly how it was trained

Evaluate on the **official test set** with the *same* readout the solve targeted: take the
time-averaged hidden spikes, project by `W_out`, argmax over the 20 classes vs. the one-hot label.
This is `mode="firing_rate"`. Because train and test use the identical feature, this is the
"honest" number for this `W_out`.


In [ ]:
acc_way1 = accuracy(net_ridge, X_test, y_test, "firing_rate")
print("WAY 1  (mean spikes @ W_out)      test acc:", round(acc_way1, 4))

## 16. Test the ridge `W_out` — WAY 2: through the output-LIF, mean membrane

Now feed the **same** `W_out` through the full spiking pipeline: input → hidden → **non-spiking
output LIF**, then use the **time-averaged output membrane** as the logits (`mode="mean_vmem"`).

**Why is this not identical to WAY 1?** The output LIF is linear and never resets, so its
time-averaged membrane is

$$\text{mean}_t\big(\text{LIF}(\text{spk}\,W_{out})\big) = \Big(\sum_t c[t]\,\text{spk}_t\Big) W_{out},$$

i.e. a **`c[t]`-weighted** average of the hidden spikes, where `c[t]` is the output neuron's
mean-over-time impulse response. WAY 1 instead uses a **uniform** average (`c[t] = 1/T`). The plot
below shows `c[t]` is far from flat — early bins count much more than late ones (they have more time
left to integrate). So the two readouts are genuinely different functions of the same spikes:
predictions mostly agree, but the accuracies need not match.

This is exactly the subtlety the two reference scripts handle differently: the *linear-readout*
script trains and tests on the uniform mean (WAY 1), whereas the *output-LIF* script builds its
ridge feature with this very `c[t]` kernel so that the closed-form solution is optimal for WAY 2.
Mixing them — fit uniform, test through the LIF — is the mismatch demonstrated here.


In [ ]:
acc_way2 = accuracy(net_ridge, X_test, y_test, "mean_vmem")
print("WAY 2  (output-LIF mean Vmem)     test acc:", round(acc_way2, 4))

# output-neuron impulse kernel c[t]: drive a unit impulse at each delay, average Vmem over time
@torch.no_grad()
def output_kernel():
    eye = torch.eye(NB_STEPS); syn = torch.zeros(NB_STEPS); mem = torch.zeros(NB_STEPS); rec = []
    for t in range(NB_STEPS):
        syn = ALPHA * syn + eye[t]; mem = BETA * mem + syn; rec.append(mem.clone())
    return torch.stack(rec).mean(dim=0)        # c[delay], length T

c = output_kernel()
plt.figure(figsize=(6, 3)); plt.plot(c.numpy())
plt.xlabel("spike arrival bin t"); plt.ylabel("c[t]  (weight in WAY 2)")
plt.title("Output-LIF time kernel: WAY 2 = (sum_t c[t]*spk_t) @ W_out  vs  WAY 1 uniform")
plt.tight_layout(); plt.show()

# how often do the two readouts pick the same class?
@torch.no_grad()
def predictions(net, X, y, mode):
    P = []
    for xb, _ in batches(X, y, BATCH_SIZE, shuffle=False):
        P.append(readout_logits(net, net(xb, need_output=(mode != "firing_rate")), mode).argmax(dim=1).cpu())
    return torch.cat(P)

p1 = predictions(net_ridge, X_test, y_test, "firing_rate")
p2 = predictions(net_ridge, X_test, y_test, "mean_vmem")
print(f"WAY1 vs WAY2 prediction agreement: {(p1 == p2).float().mean():.4f}")
print(f"kernel c[t]: first bin {c[0]:.3f}  vs  last bin {c[-1]:.3f}  (ratio {float(c[0]/c[-1]):.1f}x)")

## 17. Recap

* **Same frozen reservoir, same `W_out`** — yet WAY 1 (uniform time-average) and WAY 2 (output-LIF
  membrane mean) give different test accuracies, because the LIF imposes a non-uniform `c[t]`
  weighting over time. They are only equal if `c[t]` is flat, which it is not.
* **Ridge ⇄ mean/rate only.** The closed form needs a prediction linear in `W_out`; `max`-over-time
  is not, so `max_vmem` is reachable by **BPTT only**.
* **`firing_rate` BPTT vs. ridge** share the identical readout — comparing them isolates *gradient
  training vs. one linear solve* on the same features.
* **To approach the reference accuracies:** raise `NB_HIDDEN` to 1000 and `NB_EPOCHS` to ~200, and
  remember the decay-constant lesson — keep `SIM_DT_MS` small (long membrane memory) regardless of
  the 14 ms bin width.

To make WAY 2 *optimal* for the same `W_out`, fit ridge on the `c[t]`-weighted feature
`X[n,h] = Σ_t c[t]·spk[n,t,h]` instead of the uniform mean — then the closed-form solution targets
exactly the output-LIF mean-membrane readout.


## 18. Training with e-prop instead of BPTT — the idea

Everything above trained the reservoir with **BPTT**: PyTorch unrolls the `NB_STEPS` loop,
stores every intermediate state, and pushes the error *backward through time* through every
recurrent connection. Accurate, but memory-hungry and biologically implausible (a synapse would
need to know the future).

**e-prop** ("eligibility propagation", Bellec et al. 2020) computes a gradient **forward in time**,
using only signals locally available at each synapse. It rests on one exact fact: the gradient of
the loss w.r.t. any weight $W_{ji}$ (presynaptic $i$ → postsynaptic $j$) can be written as a sum
over time of a product of two things:

$$\frac{dE}{dW_{ji}} = \sum_t \underbrace{L_j(t)}_{\text{learning signal}}\;\cdot\;\underbrace{e_{ji}(t)}_{\text{eligibility trace}}$$

* **Eligibility trace $e_{ji}(t)$** — a *local, forward-running* quantity that remembers how much
  this synapse could have influenced neuron $j$ recently. It depends only on the synapse's own
  presynaptic spikes (passed through the neuron's two leaky filters `ALPHA`, `BETA`) and the
  postsynaptic surrogate derivative. **This part is exact.**
* **Learning signal $L_j(t)=\partial E/\partial z_j(t)$** — how much the loss cares about neuron $j$
  spiking right now. **This is the only approximated part:** e-prop keeps just the *direct* path
  from neuron $j$ to the output layer and drops the part that would re-route through future hidden
  states (the term that normally forces backprop-through-time). With **symmetric feedback** the
  broadcast weights are simply the readout weights $W_{out}$.

So e-prop = (exact local trace) × (cheap instantaneous error broadcast through `W_out`), summed
over time. No backward pass, constant memory.

### Why the trace stays cheap
A presynaptic spike reaches neuron $j$'s membrane through **two cascaded leaky filters** — first
the synaptic current (`ALPHA`), then the membrane (`BETA`) — *the very dynamics in your `hidden()`
loop*. That filter is identical for every postsynaptic $j$, so the eligibility **vector** depends
only on the **presynaptic** index $i$:

$$\zeta_i(t)=\alpha\,\zeta_i(t-1)+z^{\text{pre}}_i(t),\qquad
  \varepsilon_i(t)=\beta\,\varepsilon_i(t-1)+\zeta_i(t)$$

and the full trace factorizes as $e_{ji}(t)=\psi_j(t)\,\varepsilon_i(t)$, where
$\psi_j(t)=1/(\text{SURR\_SLOPE}\,|v_j-\theta|+1)^2$ is **exactly your `SpikeFn.backward`**.
Because of this factorization we never build the giant $[B,C,H]$ trace tensor — we keep a small
presynaptic vector and accumulate the gradient as a per-step outer product.

### What changes vs. the BPTT cells
* Same three trainable matrices (`W_in`, `W_rec`, `W_out`), same Adamax/LR/epochs — only the
  *gradient source* changes.
* The readout becomes a **leaky output** $y_k(t)=\kappa\,y_k(t-1)+\sum_j W_{out}[j,k]\,z_j(t)$ with
  **per-step** softmax cross-entropy against the constant one-hot label. Applying the loss every
  step (rather than aggregating first, like `mean_vmem`) is what keeps the learning signal *local
  in time*. The eval below uses the matching readout so numbers are comparable.
* The spike reset is detached and excluded from the trace — exactly as your BPTT code already does
  with `spk.detach()`.


## 19. e-prop helper constants

Two tiny pieces shared by everything below:

* `KAPPA` — the leak of the **output** neuron's low-pass filter. We reuse the membrane time
  constant, so `KAPPA == BETA` (a 10 ms readout memory).
* `surrogate_deriv` — the postsynaptic factor $\psi_j(t)$. It is **the same formula** as
  `SpikeFn.backward` in cell&nbsp;14; here we just call it explicitly (e-prop has no autograd
  backward pass to hook into, so we evaluate the surrogate derivative by hand).


In [ ]:
KAPPA = math.exp(-SIM_DT_MS / TAU_MEM_MS)   # leaky-readout decay (== BETA here)

def surrogate_deriv(v_minus_thresh):
    # identical to SpikeFn.backward: fast-sigmoid derivative  psi_j(t)
    return 1.0 / (SURR_SLOPE * v_minus_thresh.abs() + 1.0) ** 2

## 20. The e-prop forward pass, step by step (plain English ↔ code)

Below is the heart of the method: a **single forward sweep** over time that simultaneously runs the
network *and* builds the gradients. There is **no `loss.backward()`** — we assemble the gradient
piece by piece as we go. Here is exactly what each block of the next code cell does and why.

**Setup (before the loop).**
- We keep the usual neuron state `syn, v, z` (synaptic current, membrane, spike) plus a leaky
  readout state `y` — the same variables as `hidden()` and `output_lif()`, just gathered in one place.
- We create the **eligibility vectors**: `zeta_in/eps_in` for the input synapses and
  `zeta_rec/eps_rec` for the recurrent synapses. Each pair is the two-stage leaky filter
  ($\alpha$ then $\beta$) of the *presynaptic* spikes. We also keep `z_filt`, the $\kappa$-filtered
  hidden spikes, which is the readout's own eligibility.
- `gW_in, gW_rec, gW_out` are gradient accumulators, started at zero. `target` is the one-hot label.

**Inside the loop, per time step `t`:**

1. **Run the neuron (your exact dynamics).** `cur = x_t @ W_in + z_prev @ W_rec`, then
   `syn = ALPHA*syn + cur`, `v = BETA*v + syn`, `z = (v > THRESHOLD)`. This is identical to
   `hidden()`. We then record $\psi_j(t)$ = `surrogate_deriv(v - THRESHOLD)` **before** resetting,
   and reset with `v = v*(1-z)` (detached, just like BPTT).

2. **Advance the eligibility vectors.** `eps_in` is the double-filtered input spikes `x_t`; `eps_rec`
   is the double-filtered *previous* hidden spikes `z_prev`. These are the $\varepsilon_i(t)$ above —
   "how strongly is each presynaptic line currently echoing through the neuron's filters?"

3. **Advance the readout.** `y = KAPPA*y + z @ W_out` is the leaky output; `z_filt = KAPPA*z_filt + z`
   is the matching eligibility for `W_out`.

4. **Compute the instantaneous error & learning signal.** `pi = softmax(y)`, the output error is
   `err = pi - target`, and the per-step cross-entropy is added into `loss`. The learning signal for
   the hidden layer is this error **broadcast back through the readout weights**:
   `L = err @ W_out.t()` — the symmetric-feedback approximation. `Lpsi = L * psi` multiplies in the
   postsynaptic factor.

5. **Accumulate the gradients (the $\sum_t L\cdot e$).** Each gradient is a per-step **outer product**:
   - `gW_in  += eps_in.t()  @ Lpsi` — presynaptic eligibility (inputs) ⊗ postsynaptic $L\psi$.
   - `gW_rec += eps_rec.t() @ Lpsi` — same, for recurrent synapses.
   - `gW_out += z_filt.t()  @ err`  — the readout gradient, which is **exact** (no approximation,
     no $\psi$): filtered presynaptic spike ⊗ output error.

   The matmul sums over the batch; the loop sums over time. That is literally
   $\frac{dE}{dW_{ji}}=\sum_t L_j(t)\,\psi_j(t)\,\varepsilon_i(t)$ — built incrementally.

**After the loop**, divide by `B*T` (we averaged the loss over batch and time, so the gradients
carry the same `1/(B·T)`). Return the loss, the three gradients, and the summed logits.


In [ ]:
def eprop_forward_and_grads(net, xb, yb):
    """One forward sweep that ALSO returns e-prop gradients (no autograd).
    xb:[B,T,C] float, yb:[B] long.  Returns (loss, (gW_in,gW_rec,gW_out), logits_sum)."""
    B   = xb.shape[0]
    dev = xb.device
    W_in, W_rec, W_out = net.W_in, net.W_rec, net.W_out

    # --- neuron / readout state (same variables as hidden() + output_lif()) ---
    syn = torch.zeros(B, NB_HIDDEN,  device=dev)
    v   = torch.zeros(B, NB_HIDDEN,  device=dev)
    z   = torch.zeros(B, NB_HIDDEN,  device=dev)
    y   = torch.zeros(B, NB_OUTPUTS, device=dev)          # leaky readout

    # --- eligibility vectors: presynaptic spikes double-filtered (ALPHA then BETA) ---
    zeta_in  = torch.zeros(B, NB_INPUTS, device=dev); eps_in  = torch.zeros(B, NB_INPUTS, device=dev)
    zeta_rec = torch.zeros(B, NB_HIDDEN, device=dev); eps_rec = torch.zeros(B, NB_HIDDEN, device=dev)
    z_filt   = torch.zeros(B, NB_HIDDEN, device=dev)       # KAPPA-filtered spikes (readout eligibility)

    gW_in  = torch.zeros_like(W_in)
    gW_rec = torch.zeros_like(W_rec)
    gW_out = torch.zeros_like(W_out)

    target = torch.zeros(B, NB_OUTPUTS, device=dev)
    target[torch.arange(B), yb] = 1.0

    logits_sum = torch.zeros(B, NB_OUTPUTS, device=dev)
    loss, T = 0.0, NB_STEPS

    for t in range(T):
        x_t, z_prev = xb[:, t], z                          # presyn: input now, hidden from t-1

        # 1) neuron dynamics (identical to hidden())
        cur = x_t @ W_in + z_prev @ W_rec
        syn = ALPHA * syn + cur
        v   = BETA  * v   + syn
        z   = (v > THRESHOLD).float()
        psi = surrogate_deriv(v - THRESHOLD)               # psi_j(t)   [B,H]
        v   = v * (1.0 - z)                                # reset, detached (not in trace)

        # 2) eligibility vectors (double leaky filter of presynaptic spikes)
        zeta_in  = ALPHA * zeta_in  + x_t;    eps_in  = BETA * eps_in  + zeta_in     # [B,C]
        zeta_rec = ALPHA * zeta_rec + z_prev; eps_rec = BETA * eps_rec + zeta_rec    # [B,H]

        # 3) leaky readout + its own eligibility
        y      = KAPPA * y      + z @ W_out                # [B,O]
        z_filt = KAPPA * z_filt + z                        # [B,H]

        # 4) instantaneous error + learning signal (symmetric feedback through W_out)
        pi   = torch.softmax(y, dim=1)
        err  = pi - target                                 # output error  [B,O]
        loss = loss - (target * torch.log(pi + 1e-8)).sum(1).mean() / T
        L    = err @ W_out.t()                             # learning signal L_j(t)  [B,H]
        Lpsi = L * psi                                     # [B,H]

        # 5) accumulate gradients  (sum_t  L * psi * eps),  matmul sums over batch
        gW_in  += eps_in.t()  @ Lpsi                       # [C,H]
        gW_rec += eps_rec.t() @ Lpsi                       # [H,H]
        gW_out += z_filt.t()  @ err                        # [H,O]  (exact readout grad)
        logits_sum += y

    inv = 1.0 / (B * T)                                    # loss was averaged over batch & time
    return loss, (gW_in * inv, gW_rec * inv, gW_out * inv), logits_sum

## 21. Evaluating the e-prop network

To score fairly we read out the **same way e-prop trained**: run the leaky output `y` and sum it
over time, then argmax over the 20 classes. (Summing the per-step logits is the inference-time
counterpart of the per-step softmax loss.) This mirrors your `accuracy()` helper but with the
leaky-readout head instead of `mean_vmem`/`firing_rate`.


In [ ]:
@torch.no_grad()
def eprop_logits(net, xb):
    B = xb.shape[0]
    syn = torch.zeros(B, NB_HIDDEN, device=xb.device); v = torch.zeros_like(syn); z = torch.zeros_like(syn)
    y   = torch.zeros(B, NB_OUTPUTS, device=xb.device); out = torch.zeros_like(y)
    for t in range(NB_STEPS):
        cur = xb[:, t] @ net.W_in + z @ net.W_rec
        syn = ALPHA * syn + cur; v = BETA * v + syn
        z = (v > THRESHOLD).float(); v = v * (1.0 - z)
        y = KAPPA * y + z @ net.W_out
        out += y                                           # sum leaky output over time
    return out

@torch.no_grad()
def eprop_accuracy(net, X, y):
    correct = 0
    for xb, yb in batches(X, y, BATCH_SIZE, shuffle=False):
        correct += (eprop_logits(net, xb).argmax(1) == yb).sum().item()
    return correct / len(y)

## 22. The e-prop trainer

The training loop is deliberately the twin of `train_bptt`: same `make_net(seed)` (so it starts
from the **identical** random reservoir as the BPTT and ridge runs), same Adamax, same `LR`/epochs.
The only difference is *where the gradients come from* — instead of `loss.backward()`, we call
`eprop_forward_and_grads`, drop the returned tensors straight into each parameter's `.grad`, and let
Adamax `step()`. We set `requires_grad_(False)` because we are supplying gradients by hand.

> **Tip.** e-prop's approximate learning signal usually needs **more epochs** than BPTT to catch up,
> and often a slightly larger `LR` (try `5e-4`). To instead keep the reservoir *frozen* and train
> only the readout (a cheaper "reservoir + e-prop readout"), simply don't assign `gi`/`gr`.


In [ ]:
def train_eprop(epochs=NB_EPOCHS, seed=SEED, lr=LR, verbose_every=5):
    net = make_net(seed)                       # SAME random reservoir as the BPTT / ridge runs
    for p in net.parameters():
        p.requires_grad_(False)                # gradients are computed by hand
    opt = torch.optim.Adamax([net.W_in, net.W_rec, net.W_out], lr=lr)
    for ep in range(epochs):
        t0 = time.time()
        for xb, yb in batches(X_tr, y_tr, BATCH_SIZE, shuffle=True):
            _, (gi, gr, go), _ = eprop_forward_and_grads(net, xb, yb)
            net.W_in.grad, net.W_rec.grad, net.W_out.grad = gi, gr, go   # hand Adamax the e-prop grads
            opt.step()
            net.W_in.grad = net.W_rec.grad = net.W_out.grad = None
        if (ep + 1) % verbose_every == 0 or ep == 0:
            print(f"[eprop] epoch {ep+1:3d}/{epochs}  val_acc={eprop_accuracy(net, X_va, y_va):.4f}"
                  f"  ({time.time()-t0:.1f}s/epoch)")
    return net

net_eprop = train_eprop()
print("TEST  eprop :", round(eprop_accuracy(net_eprop, X_test, y_test), 4))